In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
station = pd.read_csv('../../archive-2/stations.csv')

In [3]:
station.head()

,StationId,StationName,City,State,Status
0,AP001,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,Active
1,AP002,"Anand Kala Kshetram, Rajamahendravaram - APPCB",Rajamahendravaram,Andhra Pradesh,NaN
2,AP003,"Tirumala, Tirupati - APPCB",Tirupati,Andhra Pradesh,NaN
3,AP004,"PWD Grounds, Vijayawada - APPCB",Vijayawada,Andhra Pradesh,NaN
4,AP005,"GVM Corporation, Visakhapatnam - APPCB",Visakhapatnam,Andhra Pradesh,Active


In [4]:
station_data_by_day = pd.read_csv('../../archive-2/station_day.csv')

In [5]:
station_coords = pd.read_csv('../../archive-2/city_coordinates.csv')

In [6]:
station_data_by_day.head()

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,AP001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,0.10,NaN,NaN
1,AP001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,0.06,184.0,Moderate
2,AP001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,0.08,197.0,Moderate
3,AP001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,0.12,198.0,Moderate
4,AP001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,0.07,188.0,Moderate


In [7]:
station_data_by_day = station_data_by_day.merge(station , on='StationId' , how='left')

In [8]:
df = station_data_by_day[['StationId' , 'Date' , 'PM2.5' , 'PM10' , 'NO' , 'NO2' , 'NOx' , 'NH3' , 'CO' , 'SO2' , 'O3' , 'Benzene' , 'Toluene' , 'Xylene' , 'AQI' , 'AQI_Bucket']]

In [9]:
df.isnull().sum()

StationId         0
Date              0
PM2.5         21625
PM10          42706
NO            17106
NO2           16547
NOx           15500
NH3           48105
CO            12998
SO2           25204
O3            25568
Benzene       31455
Toluene       38702
Xylene        85137
AQI           21010
AQI_Bucket    21010
dtype: int64

In [10]:
null_si = np.where(df['PM2.5'].isnull())
null_si

(array([   162,    163,    289, ..., 107991, 107992, 107993]),)

In [11]:
df = df.drop(columns=['Xylene'])

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 108035 entries, 0 to 108034
Data columns (total 15 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   StationId   108035 non-null  str    
 1   Date        108035 non-null  str    
 2   PM2.5       86410 non-null   float64
 3   PM10        65329 non-null   float64
 4   NO          90929 non-null   float64
 5   NO2         91488 non-null   float64
 6   NOx         92535 non-null   float64
 7   NH3         59930 non-null   float64
 8   CO          95037 non-null   float64
 9   SO2         82831 non-null   float64
 10  O3          82467 non-null   float64
 11  Benzene     76580 non-null   float64
 12  Toluene     69333 non-null   float64
 13  AQI         87025 non-null   float64
 14  AQI_Bucket  87025 non-null   str    
dtypes: float64(12), str(3)
memory usage: 14.6 MB


In [31]:
df['Date'] = pd.to_datetime(df['Date'])

In [32]:
df = df.sort_values(['StationId' , 'Date'])

In [33]:
pollutant_cols = [
    'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
    'CO', 'SO2', 'O3', 'Benzene', 'Toluene'
]

In [34]:
df[pollutant_cols] = (
    df.groupby("StationId")[pollutant_cols]
    .transform(lambda x: x.interpolate(limit_direction='both'))
)

In [40]:
df[pollutant_cols] = df[pollutant_cols].fillna(df[pollutant_cols].median())

In [36]:
AQI_col = df[['AQI' , 'AQI_Bucket']] 

In [19]:
AQI_col

,AQI,AQI_Bucket
0,NaN,NaN
1,184.0,Moderate
2,197.0,Moderate
3,198.0,Moderate
4,188.0,Moderate
...,...,...
108030,50.0,Good
108031,65.0,Satisfactory
108032,63.0,Satisfactory
108033,57.0,Satisfactory


In [20]:
df = df.dropna(subset=['AQI' , 'AQI_Bucket'])

In [21]:
df.head()

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket
1,AP001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,Moderate
2,AP001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,Moderate
3,AP001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,Moderate
4,AP001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,Moderate
5,AP001,2017-11-29,72.47,114.84,5.23,23.20,16.59,12.25,0.16,10.55,109.74,0.21,4.71,173.0,Moderate


In [22]:
station_data_by_day = station_data_by_day.merge(station_coords, on='City' , how='left' )

In [23]:
df = station_data_by_day[['StationId' , 'Date' , 'PM2.5' , 'PM10' , 'NO' , 'NO2' , 'NOx' , 'NH3' , 'CO' , 'SO2' , 'O3' , 'Benzene' , 'Toluene' , 'Xylene' , 'AQI' , 'AQI_Bucket' , 'lat' , 'lon']]

In [24]:
df.head()

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,lat,lon
0,AP001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,0.10,NaN,NaN,16.509668,80.518454
1,AP001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,0.06,184.0,Moderate,16.509668,80.518454
2,AP001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,0.08,197.0,Moderate,16.509668,80.518454
3,AP001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,0.12,198.0,Moderate,16.509668,80.518454
4,AP001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,0.07,188.0,Moderate,16.509668,80.518454


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 108035 entries, 0 to 108034
Data columns (total 18 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   StationId   108035 non-null  str    
 1   Date        108035 non-null  str    
 2   PM2.5       86410 non-null   float64
 3   PM10        65329 non-null   float64
 4   NO          90929 non-null   float64
 5   NO2         91488 non-null   float64
 6   NOx         92535 non-null   float64
 7   NH3         59930 non-null   float64
 8   CO          95037 non-null   float64
 9   SO2         82831 non-null   float64
 10  O3          82467 non-null   float64
 11  Benzene     76580 non-null   float64
 12  Toluene     69333 non-null   float64
 13  Xylene      22898 non-null   float64
 14  AQI         87025 non-null   float64
 15  AQI_Bucket  87025 non-null   str    
 16  lat         108035 non-null  float64
 17  lon         108035 non-null  float64
dtypes: float64(15), str(3)
memory usage: 17.1 MB


In [26]:
df

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,lat,lon
0,AP001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,0.10,NaN,NaN,16.509668,80.518454
1,AP001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,0.06,184.0,Moderate,16.509668,80.518454
2,AP001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,0.08,197.0,Moderate,16.509668,80.518454
3,AP001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,0.12,198.0,Moderate,16.509668,80.518454
4,AP001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,0.07,188.0,Moderate,16.509668,80.518454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108030,WB013,2020-06-27,8.65,16.46,NaN,NaN,NaN,NaN,0.69,4.36,30.59,1.32,7.26,NaN,50.0,Good,22.572646,88.363895
108031,WB013,2020-06-28,11.80,18.47,NaN,NaN,NaN,NaN,0.68,3.49,38.95,1.42,7.92,NaN,65.0,Satisfactory,22.572646,88.363895
108032,WB013,2020-06-29,18.60,32.26,13.65,200.87,214.20,11.40,0.78,5.12,38.17,3.52,8.64,NaN,63.0,Satisfactory,22.572646,88.363895
108033,WB013,2020-06-30,16.07,39.30,7.56,29.13,36.69,29.26,0.69,5.88,29.64,1.86,8.40,NaN,57.0,Satisfactory,22.572646,88.363895


In [41]:
df.isnull().sum()


StationId         0
Date              0
PM2.5             0
PM10              0
NO                0
NO2               0
NOx               0
NH3               0
CO                0
SO2               0
O3                0
Benzene           0
Toluene           0
AQI           21010
AQI_Bucket    21010
lat               0
lon               0
dtype: int64

In [39]:
df.head()

,StationId,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,AQI_Bucket,lat,lon
0,AP001,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,0.17,5.92,NaN,NaN,16.509668,80.518454
1,AP001,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,0.20,6.50,184.0,Moderate,16.509668,80.518454
2,AP001,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,0.22,7.95,197.0,Moderate,16.509668,80.518454
3,AP001,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,0.29,7.63,198.0,Moderate,16.509668,80.518454
4,AP001,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,0.17,5.02,188.0,Moderate,16.509668,80.518454


In [42]:
df.to_csv('../../datascience/data/processed/cleaned_df.csv')